# 문제 2-3 : 학생 정보 구조화 시스템

In [1]:
import os
import json
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

load_dotenv()
PROVIDER = "groq"
CFG = {
    "groq": {"key": os.getenv("GROQ_API_KEY"), "url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "openai": {"key": os.getenv("OPENAI_API_KEY"), "url": None, "model": "gpt-4o-mini"},
}
c = CFG[PROVIDER]
if not c["key"]:
    raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")

llm = ChatOpenAI(api_key=c["key"], base_url=c["url"], model=c["model"], temperature=0)


class Student(BaseModel):
    name: str = Field(description="학생의 이름")
    age: int = Field(description="학생의 나이 (숫자)")
    major: str = Field(description="학생의 전공")
    hobbies: List[str] = Field(description="학생의 취미 목록")
    goal: str = Field(description="학생의 목표")


parser = PydanticOutputParser(pydantic_object=Student)

pt = PromptTemplate(
    template="다음 학생의 자기소개에서 정보를 추출하세요.\n\n{fmt}\n\n자기소개: {intro}",
    input_variables=["intro"],
    partial_variables={"fmt": parser.get_format_instructions()},
)

chain = pt | llm | parser

intro = input("자기소개를 입력하세요 (엔터만 누르면 예시 실행): ").strip() or (
    "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. "
    "취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
)

try:
    r = chain.invoke({"intro": intro})
    print(type(r))
    print(r)
    print(json.dumps(r.model_dump(), ensure_ascii=False, indent=4))
except Exception as e:
    print("오류:", e)


<class '__main__.Student'>
name='김덕배' age=27 major='컴퓨터공학' hobbies=['게임하기', '영화보기', '코딩'] goal='훌륭한 개발자가 되는 것'
{
    "name": "김덕배",
    "age": 27,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "훌륭한 개발자가 되는 것"
}
